# Быстрый синтетический датасет для EarLoop (23-band + A/B)

Этот ноутбук генерирует **синтетический** (рандомный, но "похожий на EQ") датасет для прототипа персонализации эквалайзера.

На выходе получатся 2 CSV:

- `profiles_23band.csv` — набор кандидатных EQ-профилей: **23 значения усиления** + **6 перцептивных параметров** (bass/tilt/presence/air/lowmid/sparkle) + мета.
- `ab_pairs.csv` — A/B-пары (какие два профиля сравнивались) и **ответ пользователя** `y_a_wins`.

## Зачем это нужно
- быстро проверить пайплайн обучения предпочтений (pairwise preference learning)
- сделать демо продукта, не завися от реальных датасетов AutoEQ/SocialFX

## Что здесь "похоже на реальность"
- кривые гладкие (функции в log-frequency)
- есть **тональные архетипы**: neutral / warm / bright / v-shape / mid-forward
- есть **headroom-нормализация** (аналог preamp): максимум усиления ограничен (без клиппинга)


## 1) Параметры генерации

Твоя частотная сетка (23 полосы) подставлена напрямую.  
В колонках мы будем хранить `g_20`, `g_50`, ... (округлённые частоты), а **точная сетка** сохраняется в `meta.json`.


In [9]:
import os, json
import numpy as np
import pandas as pd

SEED = 42
OUT_DIR = "datasets/synth_quick_V1"
N_PROFILES = 300
N_PAIRS = 2500

# ТВОЯ 23-полосная сетка (Hz)
BANDS_HZ = np.array([
    20.0,
    50.0,
    83.0,
    120.0,
    159.510295,
    200.043762,
    254.048212,
    308.562725,
    383.0,
    443.863998,
    622.04351,
    798.067155,
    1000.0,
    1485.982561,
    1875.0,
    2368.080868,
    3389.648225,
    4365.363477,
    6934.260773,
    8568.995166,
    12000.0,
    14000.0,
    16000.0
], dtype=float)

rng = np.random.default_rng(SEED)
logf = np.log10(BANDS_HZ)

def col_name(hz: float) -> str:
    return f"g_{int(round(float(hz)))}"

print("bands:", len(BANDS_HZ))
print("band cols:", [col_name(hz) for hz in BANDS_HZ])


bands: 23
band cols: ['g_20', 'g_50', 'g_83', 'g_120', 'g_160', 'g_200', 'g_254', 'g_309', 'g_383', 'g_444', 'g_622', 'g_798', 'g_1000', 'g_1486', 'g_1875', 'g_2368', 'g_3390', 'g_4365', 'g_6934', 'g_8569', 'g_12000', 'g_14000', 'g_16000']


In [10]:
def gaussian(center_hz, width, amp):
    # Гладкий пик в логарифмической шкале частот (log-frequency)
    c = np.log10(center_hz)
    return amp * np.exp(-0.5 * ((logf - c) / width) ** 2)

def shelf_low(cut_hz, sharp, amp):
    # Приближённый low-shelf в log-frequency (через сигмоиду)
    x = (logf - np.log10(cut_hz)) / sharp
    return amp * (1.0 / (1.0 + np.exp(x)))

def shelf_high(cut_hz, sharp, amp):
    # Приближённый high-shelf в log-frequency (через сигмоиду)
    x = (logf - np.log10(cut_hz)) / sharp
    return amp * (1.0 / (1.0 + np.exp(-x)))

# 6 perceptual parameters we will store in the dataset
PERC_COLS = ["bass", "tilt", "presence", "air", "lowmid", "sparkle"]

def sample_params(archetype: str):
    '''
    Генерация 6-мерного перцептивного вектора.
    Смещения по архетипам имитируют типичные тональные предпочтения.
    '''
    mu = {
        "neutral":   np.array([ 0.0,  0.0,  0.0,  0.0,  0.0,  0.0]),
        "warm":      np.array([ 0.7, -0.2, -0.1, -0.4,  0.4, -0.3]),
        "bright":    np.array([-0.3,  0.1,  0.0,  0.7, -0.2,  0.6]),
        "vshape":    np.array([ 0.8,  0.0, -0.5,  0.6, -0.1,  0.4]),
        "midforward":np.array([-0.1, -0.1,  0.8, -0.1,  0.4,  0.0]),
    }[archetype]
    # jitter controls diversity inside archetype
    x = mu + rng.normal(0, 0.35, size=6)
    return x

def params_to_curve(x):
    '''
    Детерминированное отображение: перцептивные параметры → 23-band усиления (dB) на BANDS_HZ.
    Именно по этим параметрам может обучаться модель предпочтений.
    '''
    bass, tilt, presence, air, lowmid, sparkle = x

    # shapes
    tilt_shape = (logf - np.log10(1000.0))

    curve = np.zeros_like(logf, dtype=float)

    # bass (low-shelf + small bump)
    curve += shelf_low(120, 0.18, amp=4.0 * bass)
    curve += gaussian(80, 0.28, amp=1.5 * bass)

    # tilt (overall slope)
    curve += 3.0 * tilt * tilt_shape

    # low-mid warmth (200-400 Hz)
    curve += gaussian(280, 0.22, amp=3.0 * lowmid)

    # presence (2-4 kHz bump)
    curve += gaussian(3000, 0.18, amp=3.5 * presence)

    # air (high-shelf / 6-10 kHz)
    curve += shelf_high(6500, 0.20, amp=3.5 * air)

    # sparkle (very high bump around 12 kHz)
    curve += gaussian(12000, 0.18, amp=2.5 * sparkle)

    # Ограничиваем диапазон (реалистично для EQ)
    curve = np.clip(curve, -12, 12)

    # Нормализация с запасом по громкости (headroom): максимум усиления ≈ -1 dB
    headroom_db = 1.0
    preamp_db = -(float(np.max(curve)) + headroom_db)
    curve = curve + preamp_db

    return curve, preamp_db

def make_curve(archetype: str):
    x = sample_params(archetype)
    curve, preamp_db = params_to_curve(x)
    params = {k: float(v) for k, v in zip(PERC_COLS, x)}
    return curve, preamp_db, params

archetypes = ["neutral", "warm", "bright", "vshape", "midforward"]


## 3) Модель "пользователя" и генерация A/B

Мы моделируем пользователя так:

1) у пользователя есть скрытый вектор предпочтений `user_pref` в 23-band пространстве  
2) для каждой кривой считаем "полезность" (score) как скалярное произведение  
3) A/B-выбор делаем шумным: добавляем гауссов шум

Если `uA > uB`, то `y_a_wins = 1`.


In [11]:
def sample_user_mix():
    # Mixture of archetypes: a proxy for popular tonal preferences
    base = np.array([0.22, 0.22, 0.18, 0.20, 0.18])  # neutral,warm,bright,vshape,midforward
    jitter = rng.normal(0, 0.03, size=base.shape)
    p = np.clip(base + jitter, 0.05, 0.6)
    return p / p.sum()

def score(curve, user_pref_vec):
    return float(curve @ user_pref_vec)

# Archetype centroids + mixture => user_pref
arch_probs = sample_user_mix()
arch_centroids = np.stack([make_curve(a)[0] for a in archetypes], axis=0)  # curves only
mix = rng.dirichlet(alpha=5 * arch_probs)

user_pref = (mix[:, None] * arch_centroids).sum(axis=0)
user_pref = user_pref / (np.linalg.norm(user_pref) + 1e-9)

arch_probs, mix


(array([0.23619393, 0.1946113 , 0.20874641, 0.2352409 , 0.12520746]),
 array([0.33762818, 0.08842367, 0.20116153, 0.20606121, 0.1667254 ]))

## 4) Генерация профилей и сохранение CSV

- `profiles_23band.csv`: каждая строка — отдельный профиль, колонки `g_*` — значения усиления на полосах.
- `ab_pairs.csv`: пары индексов (a_id, b_id) и выбор пользователя.


In [12]:
os.makedirs(OUT_DIR, exist_ok=True)

# ---- profiles ----
profiles = []
curves = []

for i in range(N_PROFILES):
    a = rng.choice(archetypes, p=arch_probs)
    curve, preamp, params = make_curve(a)
    curves.append(curve)

    row = {"profile_id": i, "archetype": a, "preamp_db": float(preamp), **params}
    for j, hz in enumerate(BANDS_HZ):
        row[col_name(hz)] = float(curve[j])
    profiles.append(row)

profiles_df = pd.DataFrame(profiles)
curves = np.stack(curves, axis=0)

# ---- A/B pairs ----
pairs = []
noise_std = 0.35

for k in range(N_PAIRS):
    i, j = rng.choice(N_PROFILES, size=2, replace=False)
    ui = score(curves[i], user_pref) + rng.normal(0, noise_std)
    uj = score(curves[j], user_pref) + rng.normal(0, noise_std)
    y = int(ui > uj)
    pairs.append({"pair_id": k, "a_id": int(i), "b_id": int(j), "y_a_wins": int(y)})

pairs_df = pd.DataFrame(pairs)

In [13]:
# ---- split ----
idx = rng.permutation(len(pairs_df))
pairs_df["split"] = "train"
pairs_df.loc[idx[1600:2000], "split"] = "val"
pairs_df.loc[idx[2000:], "split"] = "test"

# ---- save ----
profiles_path = os.path.join(OUT_DIR, "profiles_23band.csv")
pairs_path = os.path.join(OUT_DIR, "ab_pairs.csv")
profiles_df.to_csv(profiles_path, index=False)
pairs_df.to_csv(pairs_path, index=False)

meta = {
    "bands_hz": BANDS_HZ.tolist(),
    "bands_cols": [col_name(hz) for hz in BANDS_HZ],
    "archetypes": archetypes,
    "arch_probs": arch_probs.tolist(),
    "user_mix": mix.tolist(),
    "noise_std": noise_std,
    "note": "Quick synthetic dataset with smooth archetypal EQ curves + headroom normalization."
}
with open(os.path.join(OUT_DIR, "meta.json"), "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print("Saved:")
print(" -", profiles_path, profiles_df.shape)
print(" -", pairs_path, pairs_df.shape)
print(" -", os.path.join(OUT_DIR, "meta.json"))
profiles_df.head()

Saved:
 - datasets/synth_quick_V1\profiles_23band.csv (300, 32)
 - datasets/synth_quick_V1\ab_pairs.csv (2500, 5)
 - datasets/synth_quick_V1\meta.json


,profile_id,archetype,preamp_db,bass,tilt,presence,air,lowmid,sparkle,g_20,...,g_1486,g_1875,g_2368,g_3390,g_4365,g_6934,g_8569,g_12000,g_14000,g_16000
0,0,warm,-6.221703,1.005000,-0.121742,0.137620,-0.376347,0.501192,-0.079049,-1.484424,...,-6.205469,-6.144570,-6.078300,-6.212423,-6.535455,-7.253329,-7.535341,-7.853617,-7.932036,-7.970643
1,1,vshape,-4.893288,0.688115,-0.164630,-0.723607,0.503700,0.423229,0.096959,-1.235452,...,-5.499645,-6.243776,-7.050842,-7.232092,-6.355611,-4.592447,-4.144014,-3.798668,-3.752925,-3.753665
2,2,bright,-5.653092,-0.889004,-0.017210,0.056964,0.905178,0.048929,0.877672,-9.206988,...,-5.498612,-5.367244,-5.186489,-4.849936,-4.503446,-3.061524,-2.070409,-1.007760,-1.000000,-1.215161
3,3,neutral,-1.031580,-0.161823,0.300292,-0.066957,-0.446490,-0.396651,-0.321808,-3.225021,...,-1.000000,-1.009218,-1.051379,-1.091985,-1.114932,-1.475701,-1.787990,-2.100981,-2.064385,-1.948286
4,4,warm,-4.381094,0.749849,0.041670,-0.249538,-0.344511,0.618957,-0.408271,-1.521881,...,-4.599150,-4.876564,-5.195480,-5.395789,-5.289919,-5.459421,-5.812024,-6.223923,-6.205260,-6.089370


## Как это работает (очень “сжёвано”)

1) **Архетипы** задают базовые "тональности" (warm/bright/v-shape и т.п.).  
2) **Профиль** = tilt + несколько гладких бугров + архетипные добавки.  
3) **Headroom/preamp**: сдвигаем кривую вниз так, чтобы максимум был около -1 dB (меньше риск клиппинга).  
4) **Пользователь** — смесь архетипов (это его “вкус”).  
5) **A/B**: берём две кривые и выбираем ту, у которой score выше (с шумом).

Это датасет для прототипа: он даёт реалистичные формы и стабильный пайплайн, без зависимости от AutoEQ/SocialFX.
